# 07 - R1-6 and L5 Diagnostics

This notebook is a focused diagnostic entry for two questions:

1. Why is `R1-6` still showing a high baseline voltage?
2. Why does `L5` respond strangely, especially around `R1-6 -> L5` connectivity?

It also adds a simple cache for the built `net`, so repeated analysis can load the saved object directly instead of rebuilding from raw CSVs and morphology packages every time.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from neuro_framework.models import (
    load_or_build_cached_net,
    apply_postbuild_parameter_overrides,
    build_pathway_override_rules,
    build_per_neuron_table,
    type_indices,
    run_equilibration,
    build_edge_audit_table,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
DATA_DIR = '../../mcHH/data/optic_lobe_right'
MORPH_PKG_DIR = '../../mcHH/data/optic_lobe_type_packages_v1'
ION_RULES = '../data/ion_channel_rules.csv'
SYN_RULES = '../data/synapse_rules.csv'
NT_ION_RULES = None
ROOT_ION_OVERRIDES = None

DT = 0.1
NCOMP = 2
MIN_SYN_COUNT = 3
MORPH_PROGRESS_EVERY = 5000

R16_ELEAK_SHIFT_MV = -10.0
R16_VTH_SHIFT_MV = 8.0
R16_TARGET_VTH_SHIFTS_MV = {'L1': -4.0, 'L3': -4.0}
R16_TARGET_GS_GAINS = {'L1': 1.5, 'L3': 1.25}

CACHE_DIR = '../../mcHH/data/cache'
NET_CACHE_PATH = f'{CACHE_DIR}/optic_lobe_net_type_rules_v3.pt'
FORCE_REBUILD = False
SAVE_CACHE_AFTER_BUILD = True

EQUIL_T_MS = 100.0
EQUIL_PROGRESS_EVERY = 200
TRACE_TYPES = ['R1-6', 'L5', 'L1', 'L3', 'Mi1']

print('NET_CACHE_PATH =', NET_CACHE_PATH)
print('ION_RULES      =', ION_RULES)
print('SYN_RULES      =', SYN_RULES)


In [ ]:
cache_meta = {
    'data_dir': DATA_DIR,
    'morph_pkg_dir': MORPH_PKG_DIR,
    'ion_rules': ION_RULES,
    'syn_rules': SYN_RULES,
    'dt': DT,
    'ncomp': NCOMP,
    'min_syn_count': MIN_SYN_COUNT,
}


In [ ]:
net, cache_meta_loaded, loaded_from_cache, build_elapsed = load_or_build_cached_net(
    cache_path=NET_CACHE_PATH,
    force_rebuild=FORCE_REBUILD,
    save_cache_after_build=SAVE_CACHE_AFTER_BUILD,
    cache_meta=cache_meta,
    data_dir=DATA_DIR,
    morphology_package_dir=MORPH_PKG_DIR,
    ion_rules_path=ION_RULES,
    syn_rules_path=SYN_RULES,
    nt_ion_rules_path=NT_ION_RULES,
    neuron_ion_overrides_path=ROOT_ION_OVERRIDES,
    dt=DT,
    ncomp=NCOMP,
    min_syn_count=MIN_SYN_COUNT,
    morphology_progress_every=MORPH_PROGRESS_EVERY,
)
neuron_override_rules, synapse_override_rules = build_pathway_override_rules(
    eLeak_shift_mV=R16_ELEAK_SHIFT_MV,
    v_th_shift_mV=R16_VTH_SHIFT_MV,
    target_v_th_shifts_mV=R16_TARGET_VTH_SHIFTS_MV,
    target_gs_gains=R16_TARGET_GS_GAINS,
)
net = apply_postbuild_parameter_overrides(
    net, neuron_rules=neuron_override_rules, synapse_rules=synapse_override_rules, reset_first=True
)
print('loaded_from_cache =', loaded_from_cache)
print(f'build_elapsed = {build_elapsed:.1f}s')
print(net)
print('cache_meta =', cache_meta_loaded)


In [ ]:
per_neuron_df = build_per_neuron_table(net)

display(per_neuron_df.loc[per_neuron_df['cell_type'].isin(['R1-6', 'L5'])].groupby('cell_type').agg(
    n_neurons=('root_id', 'size'),
    dominant_nt=('dominant_nt', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    param_source=('param_source', lambda s: s.mode().iloc[0] if not s.mode().empty else None),
    eLeak_mean=('eLeak_mV', 'mean'),
    gLeak_mean=('gLeak_mS_cm2', 'mean'),
).reset_index())


In [ ]:
base_init = net.init_state(batch_size=1, device=torch.device('cpu'))
base_init_soma = base_init['V'][0, net.soma_comp_idx].detach().cpu().numpy()

base_intrinsic, trace_intrinsic = run_equilibration(
    net, t_ms=EQUIL_T_MS, dt=DT, synapse_scale=0.0, trace_types=TRACE_TYPES, progress_every=EQUIL_PROGRESS_EVERY
)
base_network, trace_network = run_equilibration(
    net, t_ms=EQUIL_T_MS, dt=DT, synapse_scale=1.0, trace_types=TRACE_TYPES, progress_every=EQUIL_PROGRESS_EVERY
)

per_neuron_df['V_init_mV'] = base_init_soma
per_neuron_df['V_intrinsic_only_mV'] = base_intrinsic
per_neuron_df['V_network_on_mV'] = base_network
per_neuron_df['delta_network_from_intrinsic_mV'] = per_neuron_df['V_network_on_mV'] - per_neuron_df['V_intrinsic_only_mV']

focus_summary = per_neuron_df.loc[per_neuron_df['cell_type'].isin(['R1-6', 'L5'])].groupby('cell_type').agg(
    n_neurons=('root_id', 'size'),
    eLeak_mean=('eLeak_mV', 'mean'),
    V_init_mean=('V_init_mV', 'mean'),
    V_intrinsic_mean=('V_intrinsic_only_mV', 'mean'),
    V_network_mean=('V_network_on_mV', 'mean'),
    delta_network_from_intrinsic_mean=('delta_network_from_intrinsic_mV', 'mean'),
).reset_index()
display(focus_summary)

fig, ax = plt.subplots(figsize=(10, 4))
for ct in TRACE_TYPES:
    ax.plot(trace_intrinsic['time_ms'], trace_intrinsic[ct], '--', label=f'{ct} intrinsic-only')
    ax.plot(trace_network['time_ms'], trace_network[ct], label=f'{ct} network-on')
ax.set_xlabel('time (ms)')
ax.set_ylabel('mean soma voltage (mV)')
ax.set_title('Zero-input equilibration traces')
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
edge_df = build_edge_audit_table(net, base_network)

r16_out_summary = edge_df.loc[edge_df['pre_type'].isin(['R1-6', 'R1-R6'])].groupby(['post_type', 'param_source'], dropna=False).agg(
    n_edges=('edge_idx', 'size'),
    syn_count_sum=('syn_count', 'sum'),
    gS_mean=('gS_mS_cm2', 'mean'),
    v_th_mean=('v_th_mV', 'mean'),
    pre_baseline_mean=('pre_baseline_mV', 'mean'),
    s_inf_mean=('s_inf_baseline', 'mean'),
    drive_sum=('baseline_drive_score', 'sum'),
).reset_index().sort_values(['drive_sum', 'n_edges'], ascending=[False, False])
display(r16_out_summary.head(20))

r16_to_l5 = edge_df.loc[edge_df['pre_type'].isin(['R1-6', 'R1-R6']) & (edge_df['post_type'] == 'L5')].copy()
print('R1-6 -> L5 edge count =', len(r16_to_l5))
display(r16_to_l5[['param_source', 'fallback_target']].value_counts(dropna=False).rename('count').reset_index())
display(r16_to_l5[['syn_count', 'gS_mS_cm2', 'e_syn_mV', 'v_th_mV', 'delta_mV', 'pre_baseline_mV', 's_inf_baseline', 'baseline_drive_score']].describe())

l5_in_summary = edge_df.loc[edge_df['post_type'] == 'L5'].groupby(['pre_type', 'param_source'], dropna=False).agg(
    n_edges=('edge_idx', 'size'),
    syn_count_sum=('syn_count', 'sum'),
    gS_mean=('gS_mS_cm2', 'mean'),
    e_syn_mean=('e_syn_mV', 'mean'),
    v_th_mean=('v_th_mV', 'mean'),
    pre_baseline_mean=('pre_baseline_mV', 'mean'),
    s_inf_mean=('s_inf_baseline', 'mean'),
    drive_sum=('baseline_drive_score', 'sum'),
).reset_index().sort_values(['drive_sum', 'syn_count_sum'], ascending=[False, False])
display(l5_in_summary.head(25))

fig, ax = plt.subplots(figsize=(10, 4))
plot_df = l5_in_summary.head(12).sort_values('drive_sum', ascending=True)
ax.barh(plot_df['pre_type'] + ' | ' + plot_df['param_source'].astype(str), plot_df['drive_sum'])
ax.set_xlabel('Approx baseline drive score = gS * syn_count * s_inf')
ax.set_ylabel('Incoming pathway to L5')
ax.set_title('Largest baseline inputs to L5')
plt.tight_layout()
plt.show()


In [ ]:
# Optional export
# per_neuron_df.to_csv('r16_l5_per_neuron_diagnostics.csv', index=False)
# edge_df.to_csv('r16_l5_edge_diagnostics.csv', index=False)
# r16_to_l5.to_csv('r16_to_l5_edges.csv', index=False)
# l5_in_summary.to_csv('l5_input_summary.csv', index=False)
